# DSPy — Otimização com `KNNFewShot`

Neste notebook será demonstrado o uso do otimizador `KNNFewShot` do DSPy em um problema de **classificação binária de textos**.

Será utilizada a base **Natural Language Processing with Disaster Tweets**, disponibilizada no Kaggle. O objetivo é classificar cada tweet em uma das seguintes categorias:

* `0`: o tweet **não descreve um desastre real**;
* `1`: o tweet **descreve um desastre real**.

O experimento será dividido em duas etapas:

1. avaliar um classificador DSPy sem exemplos de demonstração (**zero-shot**);
2. utilizar `KNNFewShot` para selecionar dinamicamente exemplos semelhantes ao tweet que está sendo classificado.

A principal diferença em relação a otimizadores como `BootstrapFewShotWithRandomSearch` é que o `KNNFewShot` não procura um único conjunto fixo de demonstrações.

Para cada nova entrada, exemplos semanticamente semelhantes são recuperados do conjunto de treinamento e utilizados como contexto few-shot.

A métrica principal utilizada para comparar os dois programas será o **F1-score**.

In [1]:
import os
from dotenv import load_dotenv  # Carrega variáveis de ambiente do arquivo .env
import dspy  # Framework para otimização de prompts com Language Models
from sentence_transformers import SentenceTransformer
import pandas as pd

from typing import Literal
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

from tqdm.auto import tqdm

/home/leonardo/Documentos/github/dspy_studies/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()  # Lê variáveis do arquivo .env

True

## Setup - Configuração do Modelo

Primeiro, carregamos as variáveis de ambiente (como API key) do arquivo `.env` na raiz do projeto. Isso evita hardcoding de credenciais no código.

In [3]:
lm = dspy.LM(
    "openai/gpt-5-mini",  # Modelo OpenAI GPT-5 Mini (modelo compacto e rápido)
    api_key=os.getenv("OPENAI_API_KEY"),  # API key carregada da variável de ambiente
)

# Configura o modelo padrão para todas as operações DSPy
dspy.configure(lm=lm)

## 3. Leitura da base de dados

Será utilizada a base da competição **Natural Language Processing with Disaster Tweets**, do Kaggle.

Referência:

https://www.kaggle.com/competitions/nlp-getting-started

Para este experimento são relevantes principalmente duas colunas:

| Coluna   | Descrição                   |
| -------- | --------------------------- |
| `text`   | Texto do tweet              |
| `target` | Classe correta (`0` ou `1`) |

O problema consiste, portanto, em aprender a relação:

`text → target`


In [4]:
df = pd.read_csv('disaster_tweets.csv')
df = df.head(200)
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


## Análise da distribuição das classes

Antes da divisão dos dados, é importante verificar quantos exemplos existem de cada classe.

Além da quantidade absoluta, será analisada a proporção entre tweets classificados como `0` e `1`.

Essa análise é importante porque o **F1-score** considera conjuntamente precisão e recall e é especialmente útil quando existe algum grau de desbalanceamento entre as classes.


In [5]:
df["target"].value_counts()

target
0    103
1     97
Name: count, dtype: int64

In [6]:
df["target"].value_counts(normalize=True)

target
0    0.515
1    0.485
Name: proportion, dtype: float64

## Separação entre treino e teste

A base será dividida em:

* **80% para treinamento**;
* **20% para teste**.

O parâmetro `stratify=df["target"]` é utilizado para manter aproximadamente a mesma proporção entre as classes `0` e `1` nos dois conjuntos.

O conjunto de **treino** poderá ser utilizado pelo otimizador para selecionar exemplos de demonstração.

O conjunto de **teste**, por outro lado, será mantido separado e utilizado exclusivamente para medir a qualidade dos programas.

Essa separação evita que exemplos utilizados durante a otimização sejam empregados também na avaliação final.

In [7]:
df_train, df_test = train_test_split(
    df[["text", "target"]],
    test_size=0.20,
    random_state=42,
    stratify=df["target"],
)

print(f"Treino: {len(df_train)} exemplos")
print(f"Teste:  {len(df_test)} exemplos")

Treino: 160 exemplos
Teste:  40 exemplos


## Conversão para `dspy.Example`

O DSPy representa exemplos de treino e teste por meio da classe `dspy.Example`.

Neste problema, cada exemplo possui dois campos:

* `text`: entrada fornecida ao modelo;
* `target`: resposta esperada.

A chamada:

`with_inputs("text")`

informa explicitamente ao DSPy que `text` deve ser utilizado como entrada do programa.

Consequentemente, `target` passa a ser tratado como o **label**, ou seja, a resposta esperada para aquele exemplo.

Conceitualmente, cada registro passa a ter a seguinte estrutura:

`entrada: text → saída esperada: target`

In [8]:
def dataframe_para_dspy(dataframe):
    exemplos = []

    for _, row in dataframe.iterrows():
        exemplo = dspy.Example(
            text=row["text"],
            target=int(row["target"]),
        ).with_inputs("text")

        exemplos.append(exemplo)

    return exemplos

In [9]:
trainset = dataframe_para_dspy(df_train)
testset = dataframe_para_dspy(df_test)

print(f"Trainset DSPy: {len(trainset)}")
print(f"Testset DSPy:  {len(testset)}")

Trainset DSPy: 160
Testset DSPy:  40


In [10]:
trainset[0]

Example({'text': 'Accident center lane blocked in #SantaClara on US-101 NB before Great America Pkwy #BayArea #Traffic http://t.co/pmlOhZuRWR', 'target': 1}) (input_keys={'text'})

## Definição da tarefa com uma `Signature`

No DSPy, uma `Signature` descreve declarativamente a tarefa que será executada pelo modelo.

A `ClassificarTweet` possui:

* um `InputField` chamado `text`, contendo o tweet;
* um `OutputField` chamado `target`, contendo a classificação.

O tipo:

`Literal[0, 1]`

restringe a resposta esperada às duas classes válidas do problema.

Dessa forma, a Signature define claramente o contrato:

`texto do tweet → 0 ou 1`


In [11]:
class ClassificarTweet(dspy.Signature):
    """
    Determine se o tweet descreve um desastre real.

    Retorne:
    - 1 se o tweet estiver relacionado a um desastre real.
    - 0 caso contrário.
    """

    text: str = dspy.InputField(
        desc="Texto do tweet que deve ser classificado."
    )

    target: Literal[0, 1] = dspy.OutputField(
        desc="1 para desastre real e 0 para não desastre."
    )

In [12]:
classificador_base = dspy.Predict(ClassificarTweet)

In [13]:
def avaliar_classificador(programa, dataset, descricao="Avaliando"):
    """
    Executa um programa DSPy sobre um dataset e calcula
    métricas globais de classificação.
    """

    y_true = []
    y_pred = []

    for exemplo in tqdm(dataset, desc=descricao):

        # Executa o programa utilizando somente os campos
        # marcados como entrada pelo with_inputs(...)
        predicao = programa(**exemplo.inputs())

        # Label verdadeiro
        y_true.append(int(exemplo.target))

        # Label previsto pelo DSPy
        y_pred.append(int(predicao.target))

    # Calcula as métricas sobre todo o conjunto
    resultado = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }

    return resultado

In [14]:
resultado_base = avaliar_classificador(
    classificador_base,
    testset,
    descricao="Baseline zero-shot",
)

Baseline zero-shot: 100%|█| 40/40 [00:0


## Avaliação do baseline

O classificador inicial será executado sobre todos os exemplos do conjunto de teste.

Para cada exemplo:

1. o campo `text` é enviado ao programa;
2. o programa produz uma previsão para `target`;
3. a previsão é comparada com o `target` verdadeiro.

Ao final são calculadas as métricas globais de classificação.

Esse resultado será considerado o desempenho **antes da otimização**.


In [15]:
print(f"F1 baseline: {resultado_base['f1']:.4f}")

F1 baseline: 0.9474


In [16]:
print(f"Accuracy:  {resultado_base['accuracy']:.4f}")
print(f"Precision: {resultado_base['precision']:.4f}")
print(f"Recall:    {resultado_base['recall']:.4f}")
print(f"F1:        {resultado_base['f1']:.4f}")

Accuracy:  0.9500
Precision: 0.9474
Recall:    0.9474
F1:        0.9474


## Modelo de embeddings

O `KNNFewShot` precisa medir a similaridade entre a nova entrada e os exemplos disponíveis no conjunto de treinamento.

Para isso, cada texto será convertido em um vetor numérico, chamado **embedding**.

Neste experimento será utilizado o modelo:

`all-MiniLM-L6-v2`

da biblioteca `sentence-transformers`.

O modelo de embeddings será encapsulado por `dspy.Embedder`, permitindo que o mecanismo KNN do DSPy utilize esses vetores para procurar exemplos semanticamente semelhantes.

Conceitualmente:

```text
tweet
  ↓
modelo de embeddings
  ↓
vetor
  ↓
comparação com os vetores do trainset
  ↓
k exemplos mais semelhantes

In [17]:
# Modelo utilizado para transformar os tweets em embeddings
modelo_embeddings = SentenceTransformer("all-MiniLM-L6-v2")

# Adapta o modelo de embeddings para a interface esperada pelo DSPy
vectorizer = dspy.Embedder(
    modelo_embeddings.encode
)

Loading weights: 100%|█| 103/103 [00:00


## Otimização com `KNNFewShot`

O `KNNFewShot` é um otimizador do DSPy que utiliza o algoritmo **K-Nearest Neighbors (KNN)** para selecionar dinamicamente exemplos de demonstração.

Diferentemente de uma estratégia que escolhe um conjunto fixo de exemplos durante a compilação, o `KNNFewShot` seleciona exemplos diferentes dependendo da entrada que está sendo processada.

Neste experimento será utilizado:

```python
dspy.KNNFewShot(
    k=5,
    trainset=trainset,
    vectorizer=vectorizer,
    max_bootstrapped_demos=0,
    max_labeled_demos=5,
)
```

O parâmetro:

```python
k=5
```

indica que serão procurados os **5 exemplos mais semelhantes** à nova entrada.

O parâmetro:

```python
trainset=trainset
```

define o conjunto de exemplos no qual essa busca será realizada.

O parâmetro:

```python
vectorizer=vectorizer
```

define como os textos serão transformados em vetores para que sua similaridade possa ser calculada.

---

### Funcionamento do KNN

Durante a inicialização, os exemplos do `trainset` são convertidos em embeddings:

```text
trainset
   ↓
modelo de embeddings
   ↓
vetores dos exemplos
```

Quando um novo tweet é recebido, o mesmo processo é realizado:

```text
novo tweet
    ↓
embedding
    ↓
comparação com os embeddings do trainset
    ↓
k exemplos mais semelhantes
```

Os exemplos recuperados são então utilizados como demonstrações few-shot para o modelo.

---

### Seleção dinâmica das demonstrações

Uma característica importante do `KNNFewShot` é que as demonstrações não são necessariamente as mesmas para todas as entradas.

Por exemplo:

```text
Tweet A
   ↓
KNN
   ↓
exemplos [3, 8, 20, 25, 31]
```

enquanto outro tweet pode recuperar:

```text
Tweet B
   ↓
KNN
   ↓
exemplos [2, 7, 14, 40, 51]
```

Portanto, cada entrada pode receber um contexto few-shot específico.

---

### Configuração utilizada neste experimento

Será utilizada a configuração:

```python
max_bootstrapped_demos=0
max_labeled_demos=5
```

Com `max_bootstrapped_demos=0`, não serão geradas novas demonstrações por bootstrap.

Dessa forma, o experimento se concentra especificamente no mecanismo de recuperação do `KNNFewShot`: os exemplos utilizados como demonstração são os exemplos rotulados recuperados do próprio `trainset`.

Essa configuração também evita chamadas adicionais ao modelo durante o processo de bootstrap.

---

### Compilação

A compilação é realizada com:

```python
classificador_otimizado = optimizer.compile(
    student=classificador_base
)
```

O `trainset` não é informado ao método `compile()`, pois ele já foi fornecido ao criar o `KNNFewShot`.

Também não é necessário um `valset`, pois não existe uma busca entre diferentes programas candidatos.

---

### Antes do `KNNFewShot`

O classificador zero-shot funciona conceitualmente como:

```text
instrução
    +
novo tweet
    ↓
modelo
    ↓
classificação
```

---

### Depois do `KNNFewShot`

Com o `KNNFewShot`:

```text
novo tweet
    ↓
embedding
    ↓
busca no trainset
    ↓
k exemplos mais semelhantes
    ↓
instrução
    +
demonstrações recuperadas
    +
novo tweet
    ↓
modelo
    ↓
classificação
```

Portanto, o `KNNFewShot` **não altera os pesos do modelo de linguagem**.

A melhoria esperada ocorre porque o modelo recebe exemplos semanticamente relacionados à entrada que está sendo processada.

In [18]:
K = 5

optimizer = dspy.KNNFewShot(
    k=K,                         # Número de vizinhos semanticamente semelhantes
    trainset=trainset,           # Base utilizada para procurar os vizinhos
    vectorizer=vectorizer,       # Modelo utilizado para gerar os embeddings
    max_bootstrapped_demos=0,    # Não cria demonstrações adicionais via bootstrap
    max_labeled_demos=K,         # Utiliza os exemplos rotulados recuperados pelo KNN
)

## Compilação do programa

O método `compile()` do `KNNFewShot` recebe o programa que será utilizado como `student`.

```python
classificador_otimizado = optimizer.compile(
    student=classificador_base
)

In [19]:
classificador_otimizado = optimizer.compile(
    student=classificador_base
)

## Inspeção dos vizinhos recuperados

No `KNNFewShot`, não existe necessariamente um único conjunto fixo de demonstrações.

Os exemplos utilizados dependem da entrada que está sendo processada.

Para compreender esse comportamento, podemos fornecer um tweet de exemplo diretamente ao mecanismo KNN e verificar quais registros do `trainset` são considerados mais semelhantes.

Esses são os exemplos que servirão de base para o contexto few-shot daquela entrada específica.

In [20]:
tweet_exemplo = "A massive wildfire is spreading through the forest."

vizinhos = optimizer.KNN(
    text=tweet_exemplo
)

print(f"Número de vizinhos recuperados: {len(vizinhos)}")

Número de vizinhos recuperados: 5


In [21]:
for i, exemplo in enumerate(vizinhos, start=1):
    print(f"--- Vizinho {i} ---")
    print(f"Tweet:  {exemplo.text}")
    print(f"Target: {exemplo.target}")
    print()

--- Vizinho 1 ---
Tweet:  How the West was burned: Thousands of wildfires ablaze in California alone http://t.co/vl5TBR3wbr
Target: 1

--- Vizinho 2 ---
Tweet:  How the West was burned: Thousands of wildfires ablaze in #California alone http://t.co/iCSjGZ9tE1 #climate #energy http://t.co/9FxmN0l0Bd
Target: 1

--- Vizinho 3 ---
Tweet:  Forest fire near La Ronge Sask. Canada
Target: 1

--- Vizinho 4 ---
Tweet:  13,000 people receive #wildfires evacuation orders in California 
Target: 1

--- Vizinho 5 ---
Tweet:  #RockyFire Update => California Hwy. 20 closed in both directions due to Lake County fire - #CAfire #wildfires
Target: 1



In [22]:
tweet_exemplo = "A powerful earthquake has destroyed several buildings downtown."

vizinhos = optimizer.KNN(
    text=tweet_exemplo
)

print(f"Número de vizinhos recuperados: {len(vizinhos)}")

Número de vizinhos recuperados: 5


In [23]:
for i, exemplo in enumerate(vizinhos, start=1):
    print(f"--- Vizinho {i} ---")
    print(f"Tweet:  {exemplo.text}")
    print(f"Target: {exemplo.target}")
    print()

--- Vizinho 1 ---
Tweet:  There's an emergency evacuation happening now in the building across the street
Target: 1

--- Vizinho 2 ---
Tweet:  Set our hearts ablaze and every city was a gift And every skyline was like a kiss upon the lips @Û_ https://t.co/cYoMPZ1A0Z
Target: 0

--- Vizinho 3 ---
Tweet:  Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all
Target: 1

--- Vizinho 4 ---
Tweet:  I'm afraid that the tornado is coming to our area...
Target: 1

--- Vizinho 5 ---
Tweet:  Damage to school bus on 80 in multi car crash #BREAKING 
Target: 1



In [24]:
resultado_otimizado = avaliar_classificador(
    classificador_otimizado,
    testset,
    descricao="KNNFewShot",
)

KNNFewShot:   0%| | 0/40 [00:00<?, ?it/
  0%|            | 0/5 [00:00<?, ?it/s]

Bootstrapped 0 full traces after 0 examples for up to 1 rounds, amounting to 0 attempts.




  0%|            | 0/5 [00:00<?, ?it/s]
KNNFewShot:   5%| | 2/40 [00:00<00:03, 

Bootstrapped 0 full traces after 0 examples for up to 1 rounds, amounting to 0 attempts.



  0%|            | 0/5 [00:00<?, ?it/s]


...

Bootstrapped 0 full traces after 0 examples for up to 1 rounds, amounting to 0 attempts.


KNNFewShot:  95%|▉| 38/40 [00:02<00:00,
  0%|            | 0/5 [00:00<?, ?it/s]


Bootstrapped 0 full traces after 0 examples for up to 1 rounds, amounting to 0 attempts.



  0%|            | 0/5 [00:00<?, ?it/s]
KNNFewShot: 100%|█| 40/40 [00:02<00:00,

Bootstrapped 0 full traces after 0 examples for up to 1 rounds, amounting to 0 attempts.


In [25]:
print(f"F1 otimizado: {resultado_otimizado['f1']:.4f}")

F1 otimizado: 0.9730


## Avaliação após a aplicação do `KNNFewShot`

O classificador com `KNNFewShot` será avaliado utilizando **exatamente o mesmo conjunto de teste utilizado pelo baseline**.

Isso permite comparar:

* classificador original em modo zero-shot;
* classificador utilizando exemplos selecionados dinamicamente pelo `KNNFewShot`.

Nenhum exemplo do conjunto de teste faz parte da base utilizada pelo KNN.

Para cada exemplo do `testset`, o `KNNFewShot` procura os exemplos mais semelhantes dentro do `trainset` e utiliza esses exemplos como contexto few-shot.

Ao final serão calculados novamente:

* Accuracy;
* Precision;
* Recall;
* F1-score.

In [26]:
comparacao = pd.DataFrame(
    {
        "Modelo": [
            "Baseline (zero-shot)",
            "KNNFewShot",
        ],
        "Accuracy": [
            resultado_base["accuracy"],
            resultado_otimizado["accuracy"],
        ],
        "Precision": [
            resultado_base["precision"],
            resultado_otimizado["precision"],
        ],
        "Recall": [
            resultado_base["recall"],
            resultado_otimizado["recall"],
        ],
        "F1": [
            resultado_base["f1"],
            resultado_otimizado["f1"],
        ],
    }
)

comparacao

,Modelo,Accuracy,Precision,Recall,F1
0,Baseline (zero-shot),0.950,0.947368,0.947368,0.947368
1,KNNFewShot,0.975,1.000000,0.947368,0.972973


In [27]:
print("BASELINE")
print(
    classification_report(
        resultado_base["y_true"],
        resultado_base["y_pred"],
        digits=4,
    )
)

BASELINE
              precision    recall  f1-score   support

           0     0.9524    0.9524    0.9524        21
           1     0.9474    0.9474    0.9474        19

    accuracy                         0.9500        40
   macro avg     0.9499    0.9499    0.9499        40
weighted avg     0.9500    0.9500    0.9500        40



In [28]:
print(f"KNNFewShot (k={K})")

print(
    classification_report(
        resultado_otimizado["y_true"],
        resultado_otimizado["y_pred"],
        digits=4,
    )
)

KNNFewShot (k=5)
              precision    recall  f1-score   support

           0     0.9545    1.0000    0.9767        21
           1     1.0000    0.9474    0.9730        19

    accuracy                         0.9750        40
   macro avg     0.9773    0.9737    0.9749        40
weighted avg     0.9761    0.9750    0.9750        40



## Salvando o programa com `KNNFewShot`

O `KNNFewShot` possui um comportamento diferente de otimizadores que simplesmente armazenam demonstrações fixas no programa.

Durante a compilação, o `KNNFewShot` modifica dinamicamente o método `forward` do programa para que, a cada nova entrada:

1. seja calculado o embedding da entrada;
2. sejam recuperados os `k` exemplos mais semelhantes do conjunto de treinamento;
3. esses exemplos sejam utilizados na construção do contexto few-shot;
4. o programa seja executado utilizando essas demonstrações.

Por esse motivo, neste experimento será utilizado o **Whole Program Saving** do DSPy:

```python
classificador_otimizado.save(
    "KNNFewShot_program",
    save_program=True
)

In [29]:
# Salva o programa completo, incluindo sua arquitetura e estado
classificador_otimizado.save(
    "KNNFewShot_program",
    save_program=True
)

2026/09/07 19:06:25 WARNING dspy.primitives.base_module: Loading untrusted .pkl files can run arbitrary code, which may be dangerous. To avoid this, prefer saving using json format using module.save("module.json").


In [30]:
dspy.Predict(ClassificarTweet)

# Carrega diretamente o programa completo
classificador_carregado = dspy.load(
    "KNNFewShot_program", allow_pickle=True
)

In [31]:
tweet = "y phone battery died right before the meeting, what a disaster!"

predicao = classificador_carregado(
    text=tweet
)

print(predicao)

  0%|            | 0/5 [00:00<?, ?it/s]


Bootstrapped 0 full traces after 0 examples for up to 1 rounds, amounting to 0 attempts.
Prediction(
    target=0
)


In [32]:
# Mostra última chamada ao modelo (n=1 significa 1 última chamada)
dspy.inspect_history(n=1)





[2026-09-07T19:06:32.342058]

System message:

Your input fields are:
1. `text` (str): Texto do tweet que deve ser classificado.
Your output fields are:
1. `target` (Literal[0, 1]): 1 para desastre real e 0 para não desastre.
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## text ## ]]
{text}

Outputs will be a JSON object with the following fields.

{
  "target": "{target}        # note: the value you produce must exactly match (no extra characters) one of: 0; 1"
}
In adhering to this structure, your objective is: 
        Determine se o tweet descreve um desastre real.
        
        Retorne:
        - 1 se o tweet estiver relacionado a um desastre real.
        - 0 caso contrário.


User message:

[[ ## text ## ]]
OMG Horrible Accident Man Died in Wings of Airplane. http://t.co/xDxDPrcPnS


Assistant message:

{
  "target": 1
}


User message:

[[ ## text ## ]]
Pilot Dies In Plane 